# Produccion - Pipeline de inferencia

Clasifica asteroides nuevos (formato JPL) como PHA o no-PHA.

**Uso**: modificar `RUTA_INPUT` y ejecutar todas las celdas.
**Output**: `04_Resultados/01_Analisis/03_Produccion/reporte_produccion.csv` y `reporte_produccion.json`

Conversion a script: `jupyter nbconvert --to script 02_Produccion.ipynb`

## 1. Importar paquetes

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
warnings.filterwarnings('ignore')

## 2. Rutas y carga de artefactos

Todos los artefactos se resuelven desde la raiz del repositorio.  
**Modificar `RUTA_INPUT`** con la ruta al CSV de nuevos asteroides.

In [ ]:
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'README.md').exists():
    repo_root = repo_root.parent
if not (repo_root / 'README.md').exists():
    raise FileNotFoundError('No se encontro la raiz del repositorio (README.md).')
modelos_dir    = repo_root / '03_Modelos'
resultados_dir = repo_root / '04_Resultados'
resultados_dir.mkdir(parents=True, exist_ok=True)
RUTA_INPUT = repo_root / '01_Datos' / 'nuevo_dataset.csv'  # <- modificar aqui
print(f'Repositorio : {repo_root}')
print(f'Input       : {RUTA_INPUT}')
print(f'Resultados  : {resultados_dir}')

In [ ]:
with open(resultados_dir / 'seleccion_modelo_final.json', encoding='utf-8') as f:
    seleccion = json.load(f)
UMBRAL          = seleccion['umbral']
modelo          = joblib.load(modelos_dir / seleccion['modelo'])
encoder         = joblib.load(modelos_dir / 'pipeline_encoder.joblib')
scaler          = joblib.load(modelos_dir / 'pipeline_scaler.joblib')
SCALER_COLS     = list(scaler.feature_names_in_)
COLS_POST_SCALE = ['ad', 'n', 'per', 'per_y', 'moid', 'moid_ld']
COLS_IDS        = ['Unnamed: 0', 'spkid', 'full_name']
MEDIANAS        = dict(zip(SCALER_COLS, scaler.center_))
COLS_REQUERIDAS = set(SCALER_COLS) | {'class'}
print(f'Modelo : {seleccion["modelo"]}')
print(f'Umbral : {UMBRAL}')
print(f'Scaler : {type(scaler).__name__}  ({len(SCALER_COLS)} features)')

## 3. Funciones del pipeline

Cada funcion tiene una responsabilidad unica. Al convertir a `.py` estas funciones se exportan directamente.

In [ ]:
def validar_esquema(df):
    faltantes = COLS_REQUERIDAS - set(df.columns)
    if faltantes:
        raise ValueError(f'Columnas requeridas ausentes ({len(faltantes)}): {sorted(faltantes)}')
    print(f'Esquema OK: {df.shape[0]:,} asteroides, {df.shape[1]} columnas.')
    if 'pha' in df.columns:
        print('  Nota: columna pha ignorada (pipeline de inferencia).')

In [ ]:
def calcular_flags_calidad(df):
    num = df.copy()
    for col in ['sigma_a', 'a', 'sigma_e', 'e', 'sigma_q', 'q', 'rms']:
        if col in num.columns:
            num[col] = pd.to_numeric(num[col].replace('?', float('nan')), errors='coerce')
    flags             = pd.Series('confiable', index=df.index, dtype=str)
    incertidumbre     = pd.Series(False, index=df.index)
    ajuste_deficiente = pd.Series(False, index=df.index)
    if 'sigma_a' in num.columns and 'a' in num.columns:
        incertidumbre |= (num['sigma_a'] / num['a'].abs()) > 0.10
    if 'sigma_e' in num.columns and 'e' in num.columns:
        incertidumbre |= (num['sigma_e'] / num['e'].abs()) > 0.10
    if 'sigma_q' in num.columns and 'q' in num.columns:
        incertidumbre |= (num['sigma_q'] / num['q'].abs()) > 0.20
    if 'rms' in num.columns:
        ajuste_deficiente = num['rms'] >= 1.0
    flags[incertidumbre]                     = 'incertidumbre_orbital'
    flags[ajuste_deficiente]                 = 'ajuste_orbital_deficiente'
    flags[incertidumbre & ajuste_deficiente] = 'incertidumbre+ajuste_deficiente'
    return flags

In [ ]:
def limpiar(df):
    df_out = df.copy()
    df_out.replace('?', float('nan'), inplace=True)
    imputados = {}
    for col in SCALER_COLS:
        if col in df_out.columns:
            n_nan = int(df_out[col].isna().sum())
            if n_nan > 0:
                df_out[col] = df_out[col].astype(float).fillna(MEDIANAS[col])
                imputados[col] = n_nan
    if imputados:
        print(f'  Imputados con mediana de entrenamiento: {imputados}')
    return df_out

In [ ]:
def preprocesar(df):
    df_out = df.copy()
    df_out[['class']] = encoder.transform(df_out[['class']])
    X_ordered = df_out[SCALER_COLS].astype(float)
    X_scaled  = pd.DataFrame(scaler.transform(X_ordered), columns=SCALER_COLS, index=df_out.index)
    return X_scaled.drop(columns=COLS_POST_SCALE)

In [ ]:
def predecir(X):
    proba    = modelo.predict_proba(X)[:, 1]
    etiqueta = ['PHA' if p >= UMBRAL else 'no-PHA' for p in proba]
    return pd.DataFrame({'prob_pha': proba, 'prediccion': etiqueta}, index=X.index)

## 4. Ejecutar pipeline

Los flags de calidad se calculan antes del preprocesamiento, sobre los valores fisicos originales.

In [ ]:
df_raw = pd.read_csv(RUTA_INPUT)
print(f'Datos cargados: {df_raw.shape}')
validar_esquema(df_raw)
cols_id_presentes = [col for col in ['spkid', 'full_name'] if col in df_raw.columns]
ids        = df_raw[cols_id_presentes].copy()
flags      = calcular_flags_calidad(df_raw)
df_sin_ids = df_raw.drop(columns=[col for col in COLS_IDS if col in df_raw.columns])
df_clean   = limpiar(df_sin_ids)
X_final    = preprocesar(df_clean)
print(f'Features preparadas: {X_final.shape}')
resultados = predecir(X_final)

In [ ]:
cols_orbitales = [col for col in ['e', 'a', 'q', 'moid'] if col in df_raw.columns]
reporte = pd.concat([
    ids.reset_index(drop=True),
    df_raw[cols_orbitales].reset_index(drop=True),
    resultados.reset_index(drop=True),
    flags.rename('flag_calidad').reset_index(drop=True),
], axis=1)
reporte_pha = reporte[reporte['prediccion'] == 'PHA'].sort_values('prob_pha', ascending=False)
n_conf = int((reporte_pha['flag_calidad'] == 'confiable').sum())
print(f'Total procesados  : {len(reporte):,}')
print(f'PHAs detectados   : {len(reporte_pha)}')
print(f'  Confiables      : {n_conf}')
print(f'  Con advertencia : {len(reporte_pha) - n_conf}')
print(f'no-PHAs           : {(reporte["prediccion"] == "no-PHA").sum():,}')
if len(reporte_pha) > 0:
    print()
    print('PHAs detectados (por probabilidad descendente):')
    print(reporte_pha.to_string(index=False))

## 5. Guardar resultados

In [ ]:
ruta_csv  = resultados_dir / 'reporte_produccion.csv'
ruta_json = resultados_dir / 'reporte_produccion.json'
reporte.to_csv(ruta_csv, index=False)
n_pha_conf = int((reporte_pha['flag_calidad'] == 'confiable').sum())
resumen = {
    'modelo':              seleccion['modelo'],
    'umbral':              UMBRAL,
    'input':               str(RUTA_INPUT),
    'total_procesados':    int(len(reporte)),
    'pha_detectados':      int(len(reporte_pha)),
    'pha_confiables':      n_pha_conf,
    'pha_con_advertencia': int(len(reporte_pha)) - n_pha_conf,
    'no_pha':              int((reporte['prediccion'] == 'no-PHA').sum()),
    'distribucion_flags':  {str(k): int(v) for k,v in reporte['flag_calidad'].value_counts().items()},
}
with open(ruta_json, 'w', encoding='utf-8') as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2)
print(f'CSV  : {ruta_csv}')
print(f'JSON : {ruta_json}')
print()
print(json.dumps(resumen, indent=2))